In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# LOAD DATA
# -----------------------------
df = pd.read_csv("/content/Mouza Census-2020-Cleaned (3) (1).csv")

group_col = "Name of District"

# Drop rows where the group_col might be NaN, as they cannot be grouped meaningfully
df = df.dropna(subset=[group_col])

# -----------------------------
# DROP KNOWN ID COLUMNS (IF PRESENT)
# -----------------------------
drop_cols = ['Unnamed: 0', 'div_code', 'dist_code', 'teh_code', 'vil_code']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# -----------------------------
# CONSTANTS & MANUAL COLUMN SETTINGS
# -----------------------------

# Columns to expand into multiple frequency columns
multi_freq_cols = {
    "Status of Mouza \n1-Rural \n2-Urban \n3-Partly Urban \n4-Forest \n5-Un\x02Populated": { # Corrected column name
        1: "Rural",
        2: "Urban",
        3: "PartlyUrban",
        4: "Forest",
        5: "UnPopulated"
    },
    "Construction Type of Majority of Houses": None,   # auto-detect unique categories
    "Status Type of Majority of Streets": None,
    "Waste Management Type": None,
    "Toilet Facility in Majority of Houses": None
}

# Columns that must use mean regardless
force_mean_cols = [
    "Population Welfare Centre Available",
    "No Alternate Electricity Source"
]
# Sewerage columns to invert: new_value = 4 - old_value
invert_cols = ["BC", "BD"]

# Keywords for SUM aggregation
sum_keywords = ["total", "number", "count", "population", "area", "acres"]

# Distance detection keywords
distance_keyword = "distance"
distance_km_keyword = "km"

# Mode keywords
mode_keywords = ["electricity"]

employment_prefix = "Employment"

village_col = "Name of Village"
tehsil_col = "Name of Tehsil"

# -----------------------------
# APPLY SEWERAGE INVERSION
# -----------------------------
for col in invert_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = 4 - df[col]

# -----------------------------
# HELPER FUNCTIONS
# -----------------------------

def is_binary_series(s):
    # binary = values subset of {0,1}
    values = pd.to_numeric(s.dropna(), errors='coerce').dropna().unique()
    return set(values).issubset({0, 1}) and len(values) > 0

def pandas_mode(s):
    m = s.mode(dropna=True)
    return m.iloc[0] if len(m) > 0 else np.nan

def mean_nonzero(s):
    values = pd.to_numeric(s, errors='coerce').dropna()
    nz = values[values != 0]
    return nz.mean() if len(nz) > 0 else 0

# -----------------------------
# BUILD AGGREGATION MAP
# -----------------------------
agg_map = {}

for col in df.columns:
    if col == group_col:
        continue
    if col in [village_col, tehsil_col]:
        continue

    # Skip columns that are handled by multi_freq_cols
    if col in multi_freq_cols:
        continue

    lower = col.lower()

    # Forced mean overrides everything else
    if col in force_mean_cols:
        agg_map[col] = lambda x: pd.to_numeric(x, errors='coerce').mean(skipna=True)
        continue

    # Employment columns → mode
    if col.startswith(employment_prefix):
        agg_map[col] = lambda x, f=pandas_mode: f(x.astype(object))
        continue

    # Electricity columns → mode (except forced mean)
    if any(k in lower for k in mode_keywords) and col not in force_mean_cols:
        agg_map[col] = lambda x, f=pandas_mode: f(x.astype(object))
        continue

    # Distance columns → mean of non-zero values
    if distance_keyword in lower and distance_km_keyword in lower:
        agg_map[col] = lambda x, f=mean_nonzero: f(x)
        continue

    # SUM columns
    if any(k in lower for k in sum_keywords):
        agg_map[col] = lambda x: pd.to_numeric(x, errors='coerce').sum(skipna=True)
        continue

    # Binary columns → mean
    if is_binary_series(df[col]):
        agg_map[col] = lambda x: pd.to_numeric(x, errors='coerce').mean(skipna=True)
        continue

    # Categorical → mode
    if df[col].dtype == object:
        agg_map[col] = lambda x, f=pandas_mode: f(x.astype(object))
        continue

    # Numeric but not covered → mean
    agg_map[col] = lambda x: pd.to_numeric(x, errors='coerce').mean(skipna=True)

# -----------------------------
# INITIALIZE AGGREGATED DF WITH ALL UNIQUE DISTRICTS
# -----------------------------
all_unique_districts = df[group_col].unique()
district_agg = pd.DataFrame({group_col: all_unique_districts})

# -----------------------------
# PERFORM BASIC AGGREGATION AND MERGE
# -----------------------------
if agg_map: # Only perform aggregation if agg_map is not empty
    basic_agg_result = df.groupby(group_col).agg(agg_map).reset_index()
    district_agg = district_agg.merge(basic_agg_result, on=group_col, how="left")

# -----------------------------
# ADD UNIQUE COUNTS FOR VILLAGE & TEHSIL AND MERGE
# -----------------------------
if village_col in df.columns:
    village_counts = df.groupby(group_col)[village_col].nunique().rename("Total Villages").reset_index()
    district_agg = district_agg.merge(village_counts, on=group_col, how="left")

if tehsil_col in df.columns:
    tehsil_counts = df.groupby(group_col)[tehsil_col].nunique().rename("Total Tehsils").reset_index()
    district_agg = district_agg.merge(tehsil_counts, on=group_col, how="left")

# -----------------------------
# HANDLE MULTI-FREQUENCY COLUMNS (CORRECTED) AND MERGE
# -----------------------------
for col, fixed_map in multi_freq_cols.items():
    if col not in df.columns:
        continue

    # Compute per-district normalized frequencies
    def normalized_counts(series):
        s = pd.to_numeric(series, errors='coerce').dropna().astype(int)
        total = len(s)
        if total == 0:
            return pd.Series(dtype=float)
        return s.value_counts(normalize=True)

    # Apply normalized_counts to groups, then unstack.
    # This will result in a DataFrame indexed by group_col.
    freq_df_raw = df.groupby(group_col)[col].apply(normalized_counts).unstack(fill_value=0)

    # Reindex with all_unique_districts to ensure all districts are present, filling missing with 0.0
    freq_df_raw = freq_df_raw.reindex(all_unique_districts, fill_value=0.0).reset_index()
    freq_df_raw = freq_df_raw.rename(columns={'index': group_col})

    # If custom labels provided (e.g., "Status of Mouza")
    if fixed_map is not None:
        # Ensure all keys from fixed_map exist as columns, fill with 0.0 if not
        for key in fixed_map.keys():
            if key not in freq_df_raw.columns:
                freq_df_raw[key] = 0.0

        # Reindex to ensure consistent column order and fill with 0.0 for any missing
        freq_df_raw = freq_df_raw.reindex(columns=[group_col] + list(fixed_map.keys()), fill_value=0.0)

        rename_dict = {k: f"{col}_{v}" for k, v in fixed_map.items()}
        freq_df = freq_df_raw.rename(columns=rename_dict)

    else:
        # For auto-detected categories, rename columns (excluding the group_col itself)
        freq_df = freq_df_raw.copy()
        new_columns = [group_col] + [f"{col}_{str(c).replace(' ', '_')}" for c in freq_df.columns if c != group_col]
        freq_df.columns = new_columns

    # Merge the frequency data back to the main aggregated DataFrame
    district_agg = district_agg.merge(freq_df, on=group_col, how="left")

    # Remove the original `col` if it somehow made it into `district_agg`
    if col in district_agg.columns:
        district_agg = district_agg.drop(columns=[col])

# -----------------------------
# SAVE OUTPUT
# -----------------------------
district_agg.to_csv("district_aggregated_custom.csv", index=False)
print("Saved district_aggregated_custom.csv")

Saved district_aggregated_custom.csv
